# Data / Time Series & Recommendation Systems — 핵심 개념 및 정답 해설

원본 실습(`ts_solution.ipynb`, `RecSys_NCF.ipynb`, `RecSys_GCF_sol.ipynb`)과 `평가체크포인트.xlsx`를 기반으로 구성된 18개 핵심 문항의 정답 및 해설입니다.

## 핵심 개념 1. (a-1) MinMaxScaler 시계열 데이터 정규화 (fit_transform & transform)

### 핵심 원리 및 코드 설명
- **시계열 정규화와 데이터 누수(Data Leakage) 방지**: 신경망 학습의 수렴 속도와 안정성을 위해 주가(`Open`) 데이터를 `[0, 1]` 구간으로 스케일링합니다.
- 이때 `train_data`에는 `.fit_transform()`을 호출하여 학습 구간의 최솟값($\min$)과 최댓값($\max$)을 학습하고 변환합니다.
- 반면 미래 시점인 `test_data`에 `.fit_transform()`을 다시 호출하면 미래의 최댓값/최솟값 정보가 모델 평가에 유입되므로, 반드시 학습 데이터 기준으로 맞춰진 `scaler`의 `.transform()`만 호출해야 합니다.

### 🎯 빈칸(### 작성 필요 ###)별 출제 포인트 & 파라미터 역할
1. **빈칸 1 (`scaler.fit_transform(train_data.values)`)**:
   - **출제 포인트**: 학습 데이터로부터 스케일링 파라미터(`data_min_`, `data_max_`)를 추정(`fit`)하고 정규화된 배열로 변환(`transform`)합니다.
   - **파라미터 역할**: `train_data.values`는 `(N_train, 1)` 형상의 2D NumPy 배열입니다.
2. **빈칸 2 (`scaler.transform(test_data.values)`)**:
   - **출제 포인트**: 테스트 데이터를 재학습 없이 학습 데이터의 스케일 기준으로 변환합니다.
   - **주의사항**: `fit_transform`을 사용하지 않도록 주의합니다.

In [ ]:
# [코드]
# [0, 1] 범위로 정규화하는 MinMaxScaler 객체 생성
scaler = MinMaxScaler(feature_range=(0, 1))
# 학습 데이터의 최솟값/최댓값을 계산(fit)하고 [0, 1] 범위로 변환(transform)
train_scaled = ### scaler.fit_transform(train_data.values) ### 작성 필요 ###
# 데이터 누수 방지를 위해 테스트 데이터는 학습된 통계량으로 변환(transform)만 수행
test_scaled = ### scaler.transform(test_data.values) ### 작성 필요 ###

## 핵심 개념 2. (a-2) Sliding Window 시계열 텐서 변환 (convert_data_into_tensors)

### 핵심 원리 및 코드 설명
- **슬라이딩 윈도우(Sliding Window) 데이터 구성**: 시계열 예측 모델(LSTM, RNN, 1D-CNN)을 지도 학습하려면 연속된 시계열 $x_0, x_1, \dots, x_{T-1}$을 길이 $L$(`sequence_length`)의 입력 구간 $X_i = x_{i : i+L}$과 다음 시점 타겟 $y_i = x_{i+L, 0}$ 쌍으로 분할해야 합니다.
- 전체 길이가 $N$일 때 유효한 윈도우 개수는 $N - L$개이며, 최종 반환되는 `features`의 형상은 `(N - L, sequence_length, 1)`, `labels`의 형상은 `(N - L,)`이 됩니다.

### 🎯 빈칸(### 작성 필요 ###)별 출제 포인트 & 파라미터 역할
1. **빈칸 1 (`data_seq[i:i + sequence_length]`)**:
   - **출제 포인트**: 현재 시점 `i`부터 `sequence_length`개 타임스텝 구간을 슬라이싱하여 입력 윈도우로 추출합니다.
2. **빈칸 2 (`data_seq[i + sequence_length, 0]`)**:
   - **출제 포인트**: 입력 윈도우 직후 시점(`i + sequence_length`)의 타겟 스칼라 값(0번 컬럼)을 정답 라벨로 추출합니다.
3. **빈칸 3 (`torch.tensor(features, dtype=torch.float32)`)**:
   - **출제 포인트**: NumPy 배열을 신경망 연산이 가능한 32비트 부동소수점(`torch.float32`) 텐서로 변환합니다.

In [ ]:
# [코드]
# 과거 50개 타임스텝을 참조하여 다음 시점을 예측하도록 윈도우 크기 설정
sequence_length = 50  # Number of time steps to look back

def convert_data_into_tensors(data_seq):
  features, labels = [], []
  # 전체 시계열 길이에서 윈도우 길이를 뺀 만큼 슬라이딩하며 샘플 생성
  for i in range(len(data_seq) - sequence_length):
      # i번째부터 sequence_length 길이만큼의 입력 구간 슬라이싱: (sequence_length, 1)
      features.append(### data_seq[i:i + sequence_length] ### 작성 필요 ###)
      # 입력 구간 바로 다음 시점(i + sequence_length)의 0번 열 값을 타겟으로 추가
      labels.append(### data_seq[i + sequence_length, 0] ### 작성 필요 ###)
  # 리스트를 NumPy 배열로 일괄 변환
  features, labels = np.array(features), np.array(labels)

  # PyTorch 모델 학습을 위해 float32 텐서로 변환
  features = ### torch.tensor(features, dtype=torch.float32) ### 작성 필요 ###
  labels = torch.tensor(labels, dtype=torch.float32)
  return features, labels

## 핵심 개념 3. (a-3) TensorDataset 및 DataLoader 미니배치 구성 (to_loader)

### 핵심 원리 및 코드 설명
- **PyTorch 미니배치 파이프라인**: 텐서 형태로 변환된 `(X, y)` 쌍을 `TensorDataset`으로 래핑하면 첫 번째 차원(샘플 수 $N$)을 기준으로 인덱싱 가능한 데이터셋이 됩니다.
- 이어서 `DataLoader`를 통해 배치 크기(`batch_size=32`) 단위로 묶고, 학습용 로더는 `shuffle=True`로 에폭마다 순서를 섞어 과적합을 방지하며 테스트용 로더는 `shuffle=False`로 시간 순서를 유지합니다.

### 🎯 빈칸(### 작성 필요 ###)별 출제 포인트 & 파라미터 역할
1. **빈칸 1 (`torch.utils.data.TensorDataset(x, y)`)**:
   - **출제 포인트**: 동일한 샘플 수를 가진 특징 텐서 `x`와 라벨 텐서 `y`를 튜플 형태로 묶는 데이터셋 객체를 생성합니다.
2. **빈칸 2 (`torch.utils.data.DataLoader(dataset, batch_size, shuffle)`)**:
   - **출제 포인트**: 데이터셋으로부터 미니배치 단위로 데이터를 공급하는 반복자(iterator)를 생성합니다.

In [ ]:
# [코드]
batch_size = 32

def to_loader(x, y, batch_size, shuffle):
  # 입력 텐서 x와 라벨 텐서 y를 샘플 단위로 묶어 TensorDataset 생성
  dataset = ### torch.utils.data.TensorDataset(x, y) ### 작성 필요 ###
  # 지정된 배치 크기와 셔플 여부에 따라 미니배치를 반환하는 DataLoader 생성
  return ### torch.utils.data.DataLoader(dataset, batch_size, shuffle) ### 작성 필요 ###

# 학습 데이터는 셔플을 활성화하고, 테스트 데이터는 순서를 유지하도록 로더 생성
train_loader = to_loader(X_train, y_train, batch_size, shuffle=True)
test_loader = to_loader(X_test, y_test, batch_size, shuffle=False)

## 핵심 개념 4. (a-4) 시계열 예측을 위한 LSTMModel 클래스 구현

### 핵심 원리 및 코드 설명
- **LSTM(Long Short-Term Memory) 시계열 모델**: 게이트 메커니즘(Forget, Input, Output Gate)과 셀 상태($c_t$)를 통해 장기 시계열 의존성을 포착합니다.
- `batch_first=True`를 설정하면 입력 형상 `(B, L, input_size)`에 대해 출력 `out`이 `(B, L, hidden_size)`로 반환되며, 여기에 `nn.Linear(hidden_size, 1)`을 적용하면 각 시점의 예측값 `(B, L, 1)`을 얻습니다.

### 🎯 빈칸(### 작성 필요 ###)별 출제 포인트 & 파라미터 역할
1. **빈칸 1 (`nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)`)**:
   - **출제 포인트**: 입력 차원, 은닉 상태 차원, 스택 레이어 수 및 배치 우선 차원(`batch_first=True`)을 지정하여 순환 신경망을 초기화합니다.
2. **빈칸 2 (`nn.Linear(hidden_size, 1)`)**:
   - **출제 포인트**: `hidden_size` 차원의 은닉 표현을 1차원 스칼라 예측값으로 투영하는 출력 계층입니다.
3. **빈칸 3 (`self.lstm(x)`)**:
   - **출제 포인트**: LSTM은 `(output, (h_n, c_n))`을 반환하므로 `out, _ = self.lstm(x)`로 시퀀스 출력 텐서만 추출합니다.

In [ ]:
# [코드]
input_size = X_train.shape[-1]
num_layers = 2
hidden_size = 64

class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers):
        super(LSTMModel, self).__init__()
        # (B, L, input_size) 입력을 처리하는 다층 LSTM 레이어 정의 (batch_first=True 필수)
        self.lstm = ### nn.LSTM(input_size, hidden_size, num_layers, batch_first=True) ### 작성 필요 ###
        # 은닉 상태 벡터(hidden_size)를 1차원 예측값으로 변환하는 선형 레이어
        self.linear = ### nn.Linear(hidden_size, 1) ### 작성 필요 ###

    def forward(self, x):
        # LSTM 순전파 수행: out은 (B, L, hidden_size), 두 번째 반환값 (h_n, c_n)은 무시
        out, _ = ### self.lstm(x) ### 작성 필요 ###
        # 선형 계층을 통과시켜 (B, L, 1) 형상의 예측 텐서 반환
        return self.linear(out)

## 핵심 개념 5. (a-5) 시계열 모델 학습 및 평가 루프 (train)

### 핵심 원리 및 코드 설명
- **Many-to-One 시계열 학습 연결**: `LSTMModel(batch_x)`의 출력 형상은 `(B, seq_len, 1)`로 모든 시점의 출력을 담고 있습니다. 우리가 예측하려는 타겟 `batch_y`(형상 `(B,)`)는 윈도우 직후 시점의 값이므로, 마지막 시점의 스칼라 출력인 `[:, -1, 0]`을 인덱싱해 형상을 `(B,)`로 맞춘 뒤 `MSELoss`를 계산해야 합니다.
- 매 미니배치마다 `optimizer.zero_grad()` → `loss.backward()` → `optimizer.step()`을 수행하여 모델 파라미터를 최적화합니다.

### 🎯 빈칸(### 작성 필요 ###)별 출제 포인트 & 파라미터 역할
1. **빈칸 1 (`model(batch_x)[:, -1, 0]`)**:
   - **출제 포인트**: 3차원 출력 `(B, L, 1)`에서 마지막 타임스텝(`-1`)의 첫 번째 값(`0`)을 추출하여 타겟 `batch_y`(`(B,)`)와 차원을 일치시킵니다.
2. **빈칸 2 (`loss_fn(pred, batch_y)`)**:
   - **출제 포인트**: 예측 텐서와 정답 텐서 간의 평균 제곱 오차(MSE)를 계산합니다.
3. **빈칸 3~5 (`optimizer.zero_grad()`, `loss.backward()`, `optimizer.step()`)**:
   - **출제 포인트**: 이전 배치의 잔여 기울기를 지우고, 현재 손실에 대한 기울기를 역전파로 계산한 뒤 가중치를 갱신합니다.

In [ ]:
# [코드]
def train(model, train_loader, test_loader):
  train_hist = []
  test_hist = []
  num_epochs = 10
  # 평균 제곱 오차(MSE) 손실 함수와 Adam 옵티마이저 설정
  loss_fn = nn.MSELoss(reduction='mean')
  optimizer = optim.Adam(model.parameters(), lr=1e-3)

  for epoch in range(num_epochs):
    total_train_loss = 0.0
    total_test_loss = 0.0

    # 학습 모드 활성화
    model.train()
    for batch_x, batch_y in train_loader:
      batch_x, batch_y = batch_x.to(device), batch_y.to(device)
      # (B, L, 1) 출력에서 마지막 타임스텝(-1)의 스칼라(0) 예측값만 추출 -> (B,)
      pred = ### model(batch_x)[:, -1, 0] ### 작성 필요 ###
      # 예측값과 정답 간의 MSE 손실 계산
      loss = ### loss_fn(pred, batch_y) ### 작성 필요 ###

      # 누적된 기울기 초기화
      ### optimizer.zero_grad() ### 작성 필요 ###
      # 손실에 대한 역전파 수행
      ### loss.backward() ### 작성 필요 ###
      # 옵티마이저로 가중치 한 스텝 갱신
      ### optimizer.step() ### 작성 필요 ###

      total_train_loss += loss.item()

    avg_loss = total_train_loss / len(train_loader)
    train_hist.append(avg_loss)

    # 평가 모드 전환 및 기울기 계산 비활성화
    model.eval()
    with torch.no_grad():
      for test_x, test_y in test_loader:
        test_x, test_y = test_x.to(device), test_y.to(device)
        test_pred = model(test_x)[:, -1, 0]
        test_loss = loss_fn(test_pred, test_y)
        total_test_loss += test_loss.item()

    avg_test_loss = total_test_loss / len(test_loader)
    test_hist.append(avg_test_loss)

## 핵심 개념 6. (a-6) 시계열 예측 성능 평가 지표 계산 (RMSE & MAPE)

### 핵심 원리 및 코드 설명
- **회귀 및 시계열 평가 지표 (RMSE & MAPE)**:
  - **RMSE**: $\sqrt{\frac{1}{N}\sum (y_i - \hat{y}_i)^2}$로, 오차를 제곱하므로 큰 이상치 오차에 민감하며 원래 타겟과 동일한 단위를 가집니다.
  - **MAPE**: $\frac{1}{N}\sum \left|\frac{y_i - \hat{y}_i}{y_i}\right|$로, 실제값 대비 상대적 백분율 오차를 측정하여 스케일에 무관한 성능 비교가 가능합니다.

### 🎯 빈칸(### 작성 필요 ###)별 출제 포인트 & 파라미터 역할
1. **빈칸 1 (`y_hat[:, -1, 0]`)**:
   - **출제 포인트**: `(N, seq_len, 1)` 형상의 모델 출력에서 각 샘플의 마지막 타임스텝 예측값을 1차원 텐서 `(N,)`로 추출합니다.
2. **빈칸 2 (`np.sqrt(mean_squared_error(y_test, test_predictions))`)**:
   - **출제 포인트**: MSE를 구한 뒤 제곱근(`np.sqrt`)을 취해 RMSE를 계산합니다.
3. **빈칸 3 (`mean_absolute_percentage_error(y_test, test_predictions)`)**:
   - **출제 포인트**: `sklearn.metrics`의 MAPE 함수를 호출하여 상대 오차를 계산합니다.

In [ ]:
# [코드]
def test(model, X_test, y_test):
  # 평가 모드 전환
  model.eval()
  with torch.no_grad():
    X_test = X_test.to(device)                     # (N, seq_len, 1)
    y_hat = model(X_test)                          # (N, seq_len, 1)
    # 마지막 시점(-1)의 스칼라 예측값(0) 추출
    test_predictions = ### y_hat[:, -1, 0] ### 작성 필요 ###

  # scikit-learn 지표 계산을 위해 CPU NumPy 배열로 변환
  test_predictions = test_predictions.cpu().numpy()
  y_test = y_test.cpu().numpy()

  # MSE에 제곱근을 취하여 RMSE 계산
  rmse = ### np.sqrt(mean_squared_error(y_test, test_predictions)) ### 작성 필요 ###
  # 평균 절대 백분율 오차(MAPE) 계산
  mape = ### mean_absolute_percentage_error(y_test, test_predictions) ### 작성 필요 ###

  return rmse, mape

## 핵심 개념 7. (a-7) 1D CNN 기반 시계열 모델 (Conv1DModel) 차원 변환 및 순전파

### 핵심 원리 및 코드 설명
- **`nn.Conv1d`의 텐서 축 규격**: RNN/LSTM(`batch_first=True`)과 선형 계층(`nn.Linear`)은 `(Batch, Seq_len, Feature)` 순서를 사용하지만, PyTorch의 `nn.Conv1d`는 채널(Feature) 차원이 시퀀스 길이 앞에 오는 `(Batch, Feature, Seq_len)` 형상을 입력으로 받습니다.
- 따라서 합성곱 연산 전에 `x.transpose(1, 2)`로 축을 교환하고, `self.conv1d(x)` 통과 후 다시 `x.transpose(1, 2)`로 `(Batch, Seq_len', hidden_size)`로 복원해야 마지막 `self.fc(x)`가 채널 축에 선형 변환을 올바르게 적용할 수 있습니다.

### 🎯 빈칸(### 작성 필요 ###)별 출제 포인트 & 파라미터 역할
1. **빈칸 1 (`nn.Conv1d(in_channels=input_size, out_channels=hidden_size, kernel_size=2, stride=1)`)**:
   - **출제 포인트**: 시계열 로컬 패턴을 추출하는 1D 합성곱 레이어를 초기화합니다.
2. **빈칸 2 (`x.transpose(1, 2)`)**:
   - **출제 포인트**: `(B, L, C)` 텐서를 `Conv1d` 입력 규격인 `(B, C, L)`로 변환하고, 합성곱 이후 다시 `(B, L', H)`로 되돌립니다.

In [ ]:
# [코드]
class Conv1DModel(nn.Module):
  def __init__(self, input_size, hidden_size):
    super(Conv1DModel, self).__init__()
    # 입력 채널 input_size, 출력 채널 hidden_size, 커널 크기 2, 스트라이드 1인 1D 합성곱 정의
    self.conv1d = ### nn.Conv1d(in_channels=input_size, out_channels=hidden_size, kernel_size=2, stride=1) ### 작성 필요 ###
    # 은닉 채널(hidden_size)을 1차원 출력으로 매핑하는 선형 레이어
    self.fc = nn.Linear(hidden_size, 1)

  def forward(self, x):
    # (B, L, C) -> (B, C, L)로 차원 교환하여 Conv1d 입력 형식에 맞춤
    x = ### x.transpose(1, 2) ### 작성 필요 ###
    x = self.conv1d(x)
    # (B, H, L') -> (B, L', H)로 다시 차원을 복원하여 Linear 계층 입력에 맞춤
    x = x.transpose(1, 2)
    return self.fc(x)

## 핵심 개념 8. (a-8) 다중 스텝 예측(Seq2Seq)을 위한 시퀀스 생성 (create_enc_dec_sequences)

### 핵심 원리 및 코드 설명
- **다중 스텝(Multi-Step) 슬라이딩 윈도우**: 단일 시점 예측과 달리 Seq2Seq(Encoder-Decoder) 모델은 길이 $L_{in}$(`sequence_length=50`)의 과거 시퀀스를 입력받아 길이 $L_{out}$(`target_len=10`)의 미래 시퀀스 구간을 한 번에 예측합니다.
- 따라서 인덱스 `i`에서 시작하는 샘플 하나가 차지하는 총 길이는 `sequence_length + target_len`이므로, 유효한 시작 인덱스 범위는 `range(len(data) - sequence_length - target_len)`이 됩니다.

### 🎯 빈칸(### 작성 필요 ###)별 출제 포인트 & 파라미터 역할
1. **빈칸 1 (`len(data) - sequence_length - target_len`)**:
   - **출제 포인트**: 입력 시퀀스와 미래 타겟 시퀀스가 배열 끝을 벗어나지 않도록 하는 루프 상한값입니다.
2. **빈칸 2 (`data[i + sequence_length : i + sequence_length + target_len]`)**:
   - **출제 포인트**: 입력 윈도우가 끝나는 `i + sequence_length` 시점부터 `target_len` 길이만큼의 미래 구간을 타겟 시퀀스로 슬라이싱합니다.

In [ ]:
# [코드]
sequence_length = 50
target_len = 10

def create_enc_dec_sequences(data):
    features, labels = [], []
    # 입력 길이(sequence_length)와 출력 길이(target_len)를 모두 확보할 수 있는 범위까지 반복
    for i in range(### len(data) - sequence_length - target_len ### 작성 필요 ###):
        # 과거 sequence_length 구간을 인코더 입력으로 추가
        features.append(data[i:i + sequence_length])
        # 직후 target_len 구간을 디코더 타겟 시퀀스로 추가
        labels.append(### data[i + sequence_length : i + sequence_length + target_len] ### 작성 필요 ###)

    features = np.array(features, dtype=np.float32)
    labels = np.array(labels, dtype=np.float32)

    features = torch.tensor(features, dtype=torch.float32)
    labels = torch.tensor(labels, dtype=torch.float32)
    return features, labels

## 핵심 개념 9. (a-9) Encoder-Decoder RNN (RNNRNN) 자기회귀 디코딩 순전파

### 핵심 원리 및 코드 설명
- **Seq2Seq 자기회귀(Autoregressive) 추론**:
  1. **인코딩**: `h = self.encoder(source)`를 통해 과거 50개 시점의 정보를 은닉 상태 `h`(`(num_layers, B, hidden_size)`)에 요약합니다.
  2. **초기 디코더 입력**: 입력 시퀀스의 가장 마지막 관측값 `source[:, -1, :]`(`(B, input_size)`)에 타임스텝 차원을 추가한 `unsqueeze(1)`(`(B, 1, input_size)`)을 디코더의 첫 입력으로 넣습니다.
  3. **자기회귀 루프**: 매 시점 $t$마다 디코더가 예측한 `out`을 다음 시점 $t+1$의 입력 `input = out`으로 피드백하며 `target_len`번 반복한 뒤, `torch.stack(predictions, dim=1)`로 `(B, target_len, input_size)` 출력 텐서를 완성합니다.

### 🎯 빈칸(### 작성 필요 ###)별 출제 포인트 & 파라미터 역할
1. **빈칸 1 (`self.encoder(source)`)**:
   - **출제 포인트**: 입력 시퀀스 전체를 인코더에 통과시켜 요약된 컨텍스트 은닉 상태 `h`를 얻습니다.
2. **빈칸 2 (`source[:, -1, :].unsqueeze(1)`)**:
   - **출제 포인트**: 마지막 시점 데이터를 추출하고 RNN 입력 규격 `(B, 1, C)`를 맞추기 위해 1번 축을 추가합니다.
3. **빈칸 3 (`self.decoder(input, h)`)**:
   - **출제 포인트**: 디코더에 현재 입력과 이전 은닉 상태를 전달하여 1스텝 예측값 `out`과 새 은닉 상태 `h`를 얻습니다.
4. **빈칸 4 (`torch.stack(predictions, dim=1)`)**:
   - **출제 포인트**: 리스트에 저장된 `target_len`개의 `(B, C)` 텐서들을 시간 축(`dim=1`)으로 결합하여 `(B, target_len, C)` 텐서를 생성합니다.

In [ ]:
# [코드]
# [참고 컴포넌트 정의 - 세부 로직 비공개]
# ### 공개 제한 ###
class EncoderRNN(nn.Module):
  def __init__(self, input_size, hidden_size, num_layers):
    super(EncoderRNN, self).__init__()
    self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

  def forward(self, x):
    _, h = self.rnn(x)
    return h

class DecoderRNN(nn.Module):
  def __init__(self, input_size, hidden_size, num_layers):
    super(DecoderRNN, self).__init__()
    self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
    self.fc = nn.Linear(hidden_size, input_size)

  def forward(self, x, h):
    out, h = self.rnn(x, h)
    out = self.fc(out)
    return out, h

class RNNRNN(nn.Module):
  def __init__(self, input_size, hidden_size, num_layers):
    super(RNNRNN, self).__init__()
    self.encoder = EncoderRNN(input_size, hidden_size, num_layers)
    self.decoder = DecoderRNN(input_size, hidden_size, num_layers)

  def forward(self, source, target_len):
    # 인코더를 통해 입력 시계열로부터 문맥 은닉 상태 h 추출
    h = ### self.encoder(source) ### 작성 필요 ###
    predictions = []
    # 입력 시퀀스의 마지막 시점(-1) 값을 디코더의 첫 입력 (B, 1, C)로 설정
    input = ### source[:, -1, :].unsqueeze(1) ### 작성 필요 ###
    for t in range(target_len):
      # 디코더를 1스텝 실행하여 예측 출력 out과 갱신된 은닉 상태 h 획득
      out, h = ### self.decoder(input, h) ### 작성 필요 ###
      predictions.append(out.squeeze(1))
      # 현재 시점의 예측 출력 out을 다음 시점의 입력으로 사용 (자기회귀)
      input = out
    # 스텝별 예측 리스트를 시간 차원(dim=1)으로 쌓아 (B, target_len, C) 텐서 생성
    outputs = ### torch.stack(predictions, dim=1) ### 작성 필요 ###

    return outputs

## 핵심 개념 10. (b-1) MovieLens 커스텀 Dataset 클래스 구현 (MovieLens)

### 핵심 원리 및 코드 설명
- **PyTorch Custom Dataset 규격**: `DataLoader`와 연동되는 데이터셋 클래스는 전체 데이터 개수를 반환하는 `__len__`과, 주어진 인덱스 `item`의 샘플을 반환하는 `__getitem__`을 구현해야 합니다.
- 임베딩 레이어(`nn.Embedding`)의 입력 인덱스로 사용되는 `users`와 `movies`는 반드시 정수형 텐서(`torch.long`)여야 합니다.

### 🎯 빈칸(### 작성 필요 ###)별 출제 포인트 & 파라미터 역할
1. **빈칸 1 (`len(self.users)`)**:
   - **출제 포인트**: 데이터셋의 총 상호작용(평점) 샘플 개수를 반환합니다.
2. **빈칸 2 (`torch.tensor(users, dtype = torch.long).to(device)`)**:
   - **출제 포인트**: 인덱싱된 사용자 ID를 `nn.Embedding` 조회가 가능한 `torch.long` 텐서로 변환하여 대상 디바이스로 이동시킵니다.

In [ ]:
# [코드]
class MovieLens:
  def __init__(self, users, movies, ratings):
    self.users = users
    self.movies = movies
    self.ratings = ratings

  def __len__(self):
    # 전체 데이터셋의 샘플 개수 반환
    return ### len(self.users) ### 작성 필요 ###

  def __getitem__(self,item):
    '''
    item = randomly selected indexes for (user,item) pairs
    '''
    users = self.users[item]
    movies = self.movies[item]
    ratings = self.ratings[item]
    # 임베딩 레이어 입력을 위해 torch.long 타입 텐서로 변환 후 디바이스로 이동
    return {'users': ### torch.tensor(users, dtype = torch.long).to(device) ### 작성 필요 ###,
            'movies': torch.tensor(movies, dtype = torch.long).to(device),
            'ratings': torch.tensor(ratings, dtype=torch.long).to(device)}

## 핵심 개념 11. (b-2) LabelEncoder 연속 인덱스 변환 및 층화 분할 (train_test_split)

### 핵심 원리 및 코드 설명
- **희소 ID의 연속 인덱스 인코딩 (`LabelEncoder`)**: 원본 MovieLens의 `movieId`는 중간에 빈 번호가 많아 최댓값 기준으로 `nn.Embedding`을 생성하면 메모리가 낭비됩니다. `LabelEncoder.fit_transform()`을 사용하면 고유 ID들을 `0 ~ (고유 개수 - 1)` 범위의 연속 정수로 매핑할 수 있습니다.
- **층화 추출 (`stratify`)**: `train_test_split`에서 `stratify=df.rating.values`를 지정하면 학습 셋과 테스트 셋의 평점(0.5~5.0) 분포 비율이 원본과 동일하게 유지됩니다.

### 🎯 빈칸(### 작성 필요 ###)별 출제 포인트 & 파라미터 역할
1. **빈칸 1 (`lbl_user.fit_transform(df.userId.values)`)**:
   - **출제 포인트**: 고유 사용자 ID를 학습(`fit`)하고 `0`부터 시작하는 연속 정수 배열로 변환(`transform`)합니다.
2. **빈칸 2 (`model_selection.train_test_split(df, test_size=0.1, random_state=42, stratify=df.rating.values)`)**:
   - **출제 포인트**: 전체 데이터를 9:1로 분할하면서 `stratify` 인자로 타겟 평점 비율을 보존합니다.

In [ ]:
# [코드]
# 사용자 ID와 영화 ID를 0부터 시작하는 연속 정수로 매핑하기 위한 LabelEncoder 생성
lbl_user = preprocessing.LabelEncoder()
lbl_movie = preprocessing.LabelEncoder()

# 원본 userId와 movieId를 연속된 정수 인덱스로 변환
df.userId = ### lbl_user.fit_transform(df.userId.values) ### 작성 필요 ###
df.movieId = lbl_movie.fit_transform(df.movieId.values)

# 평점(rating) 분포를 유지(stratify)하며 학습용과 테스트용 데이터프레임을 9:1 비율로 분할
df_train, df_test = ### model_selection.train_test_split(df, test_size=0.1, random_state=42, stratify=df.rating.values) ### 작성 필요 ###

train_dataset = MovieLens(users = df_train.userId.values, movies = df_train.movieId.values, ratings = df_train.rating.values)
test_dataset = MovieLens(users = df_test.userId.values, movies = df_test.movieId.values, ratings = df_test.rating.values)

## 핵심 개념 12. (b-3) Neural Collaborative Filtering (NCF) 모델 구현

### 핵심 원리 및 코드 설명
- **Neural Collaborative Filtering (NCF)**: 전통적인 행렬 분해(Matrix Factorization)가 사용자·아이템 잠재 벡터의 단순 내적(선형 결합)만 사용하는 한계를 극복하기 위해, 두 임베딩 벡터 $e_u \in \mathbb{R}^{32}$와 $e_i \in \mathbb{R}^{32}$를 채널 축(`dim=1`)으로 이어 붙인 `[e_u || e_i]`($\in \mathbb{R}^{64}$)를 비선형 MLP(`Linear(64, 32) -> ReLU -> Linear(32, 1)`)에 통과시켜 복잡한 비선형 상호작용을 학습합니다.

### 🎯 빈칸(### 작성 필요 ###)별 출제 포인트 & 파라미터 역할
1. **빈칸 1 (`nn.Embedding(n_users, 32)`)**:
   - **출제 포인트**: 총 `n_users`명의 사용자를 각 32차원 학습 가능 벡터로 변환하는 룩업 테이블을 생성합니다.
2. **빈칸 2 (`nn.Linear(64, 32)`)**:
   - **출제 포인트**: 사용자 임베딩(32)과 영화 임베딩(32)이 결합된 64차원 입력을 32차원 은닉 특징으로 변환합니다.
3. **빈칸 3 (`torch.cat([user_embedding, movie_embedding], dim = 1)`)**:
   - **출제 포인트**: 배치 차원(`dim=0`)은 유지하고 임베딩 특징 차원(`dim=1`)을 따라 두 `(B, 32)` 텐서를 `(B, 64)`로 결합합니다.

In [ ]:
# [코드]
class Neural_Collaborative_Filtering(nn.Module):
  def __init__(self, n_users, n_movies):
    super().__init__()
    # 사용자와 영화를 각각 32차원 잠재 벡터(latent vector)로 변환하는 임베딩 계층
    self.user_embedding = ### nn.Embedding(n_users, 32) ### 작성 필요 ###
    self.movie_embedding = nn.Embedding(n_movies, 32)

    # 결합된 64차원(32 + 32) 벡터로부터 평점을 예측하는 2층 신경망(MLP)
    self.fc1 = ### nn.Linear(64,32) ### 작성 필요 ###
    self.relu = nn.ReLU()
    self.fc2= nn.Linear(32,1)

  def forward(self, users, movies, ratings = None):
    user_embedding = self.user_embedding(users)
    movie_embedding = self.movie_embedding(movies)
    # 사용자 임베딩과 영화 임베딩을 특징 차원(dim=1)으로 연결 -> (B, 64)
    input_embedding = ### torch.cat([user_embedding, movie_embedding], dim = 1) ### 작성 필요 ###

    hidden_feature = self.fc1(input_embedding)
    hidden_feature = self.relu(hidden_feature)

    # 최종 예측 평점 (B, 1) 산출
    output = self.fc2(hidden_feature)

    return output

## 핵심 개념 13. (b-4) NCF 평점 회귀 학습 및 RMSE 평가

### 핵심 원리 및 코드 설명
- **명시적 피드백(Explicit Feedback) 평점 회귀 평가**: 평점 예측 모델의 정확도는 실제 평점 $r_{u,i}$와 예측 평점 $\hat{r}_{u,i}$ 사이의 평균 제곱근 오차 $\text{RMSE} = \sqrt{\frac{1}{|T|}\sum_{(u,i)\in T} (r_{u,i} - \hat{r}_{u,i})^2}$로 측정합니다.
- 모델 출력 `(B, 1)`에서 `squeeze(axis=1)`로 불필요한 1차원을 제거해 파이썬 리스트로 누적한 뒤 `mean_squared_error`와 `np.sqrt`를 차례로 적용합니다.

### 🎯 빈칸(### 작성 필요 ###)별 출제 포인트 & 파라미터 역할
1. **빈칸 1 (`model(batched_data['users'],batched_data['movies'])`)**:
   - **출제 포인트**: 미니배치의 사용자 인덱스와 영화 인덱스를 모델에 전달하여 평점 예측 텐서를 얻습니다.
2. **빈칸 2 (`mean_squared_error(target_rating_list, model_output_list)`)**:
   - **출제 포인트**: 전체 테스트 샘플의 실제 평점과 예측 평점 간의 MSE를 계산합니다.
3. **빈칸 3 (`np.sqrt(mse)`)**:
   - **출제 포인트**: MSE에 제곱근을 취해 평점 단위와 동일한 RMSE를 산출합니다.

In [ ]:
# [코드]
from sklearn.metrics import mean_squared_error

model_output_list = []
target_rating_list = []

# 평가 모드 전환
model.eval()

with torch.no_grad():
  for i, batched_data in enumerate(test_loader):
    # 미니배치의 사용자 ID와 영화 ID를 입력하여 예측 평점 산출
    model_output = ### model(batched_data['users'],batched_data['movies']) ### 작성 필요 ###
    # (B, 1) 텐서를 CPU NumPy로 옮긴 뒤 1번 축을 제거(squeeze)하여 리스트에 추가
    model_output_batch = model_output.cpu().numpy().squeeze(axis=1).tolist()
    model_output_list += (model_output_batch)

    target_rating = batched_data['ratings']
    target_rating_batch = target_rating.cpu().numpy().tolist()
    target_rating_list += target_rating_batch

# 전체 테스트 셋에 대한 평균 제곱 오차(MSE) 계산
mse = ### mean_squared_error(target_rating_list, model_output_list) ### 작성 필요 ###
# MSE에 제곱근을 취하여 최종 RMSE 계산
rms = ### np.sqrt(mse) ### 작성 필요 ###
print(f"rms: {rms}")

## 핵심 개념 14. (b-5) 추천 시스템 Top-K 평가 지표 (Precision@K & Recall@K) 계산

### 핵심 원리 및 코드 설명
- **Precision@K와 Recall@K의 정의**:
  - 기준 임계값(`threshold = 3.5`) 이상인 실제 평점을 가진 아이템을 '관련(relevant) 아이템'으로 정의합니다.
  - 예측 평점(`est`)이 높은 순서대로 상위 $K$개(`recommended_k = user_ratings[:k]`)를 추천했을 때:
    $$\text{Precision@K} = \frac{|\text{Top-K} \cap \text{Relevant}|}{K}, \quad \text{Recall@K} = \frac{|\text{Top-K} \cap \text{Relevant}|}{|\text{Relevant}|}$$

### 🎯 빈칸(### 작성 필요 ###)별 출제 포인트 & 파라미터 역할
1. **빈칸 1 (`user_ratings[:k]`)**:
   - **출제 포인트**: 예측 평점 내림차순으로 정렬된 리스트에서 상위 `k`개 추천 결과만 슬라이싱합니다.
2. **빈칸 2 (`n_real_relevant_in_top_k / 10`)**:
   - **출제 포인트**: 추천한 10개 아이템 중 실제로 선호한(`true_r >= 3.5`) 아이템의 비율(Precision@10)을 계산합니다.
3. **빈칸 3 (`n_real_relevant_in_top_k / n_real_relevant`)**:
   - **출제 포인트**: 사용자가 실제로 선호한 전체 아이템(`n_real_relevant`) 중 상위 K개 안에 포함된 비율(Recall@K)을 계산합니다.

In [ ]:
# [코드]
with torch.no_grad():
  precisions = dict()
  recalls = dict()

  # 상위 추천 개수 k와 관련 아이템 판단 임계값(3.5점 이상) 설정
  k= 10
  threshold = 3.5 # relevant item criterion

  for user_id, user_ratings in user_est_true.items():

    # 사용자가 실제로 3.5점 이상 준 전체 관련 아이템 수 (Recall@k의 분모)
    n_real_relevant= sum((true_r >= threshold) for (_, true_r) in user_ratings)

    # 예측 평점(x[0])이 높은 순서대로 내림차순 정렬
    user_ratings.sort(key=lambda x: x[0], reverse =True)

    # 상위 k개 추천 결과 추출
    recommended_k = ### user_ratings[:k] ### 작성 필요 ###

    # 상위 k개 추천 아이템 중 실제 평점이 3.5 이상인 적중(Hit) 개수 합산
    n_real_relevant_in_top_k = sum((true_r >= threshold) for (est, true_r) in recommended_k)

    # Precision@k 계산 (추천된 10개 중 실제 관련 아이템 비율)
    precisions[user_id] = ### n_real_relevant_in_top_k / 10 ### 작성 필요 ###
    # Recall@k 계산 (전체 관련 아이템 중 추천된 상위 k개에 포함된 비율)
    if n_real_relevant:
      recalls[user_id] = ### n_real_relevant_in_top_k / n_real_relevant ### 작성 필요 ###
    else:
      recalls[user_id] = 0

## 핵심 개념 15. (c-1) 사용자-아이템 이분 그래프 Edge Index 생성 (create_edge_index)

### 핵심 원리 및 코드 설명
- **이분 그래프(Bipartite Graph) 노드 인덱싱**: 그래프 신경망(NGCF)에서는 전체 노드를 단일 임베딩 행렬 $E \in \mathbb{R}^{(N_{user} + N_{item}) \times d}$로 관리합니다.
- 사용자 노드가 `0`번부터 `num_users - 1`번까지를 차지하므로, 영화 아이템 노드는 `num_users`만큼 오프셋을 더한 `movieId + num_users`로 인덱싱해야 서로 다른 노드로 구분됩니다.

### 🎯 빈칸(### 작성 필요 ###)별 출제 포인트 & 파라미터 역할
1. **빈칸 1 (`row['movieId'] + num_users`)**:
   - **출제 포인트**: 아이템 노드의 인덱스가 사용자 노드 인덱스 뒤에 이어지도록 `num_users` 오프셋을 더합니다.
2. **빈칸 2 (`torch.tensor([src, dst], dtype=torch.long)`)**:
   - **출제 포인트**: `(2, num_edges)` 형상의 정수형(`torch.long`) 엣지 인덱스 텐서를 생성합니다.

In [ ]:
# [코드]
def create_edge_index(df, rating_threshold=1.0):
    src, dst = [], []
    for _, row in df.iterrows():
        # 기준 평점 이상인 상호작용만 그래프의 엣지로 연결
        if row['rating'] >= rating_threshold:
            src.append(row['userId'])
            # 아이템 노드 인덱스가 사용자 노드 인덱스 뒤에 위치하도록 num_users 오프셋 합산
            dst.append(### row['movieId'] + num_users ### 작성 필요 ###)
    # [출발 노드 리스트, 도착 노드 리스트]를 (2, E) 형상의 long 텐서로 반환
    return ### torch.tensor([src, dst], dtype=torch.long) ### 작성 필요 ###

edge_index = create_edge_index(rating_df)

## 핵심 개념 16. (c-2) NGCF 그래프 컨볼루션 레이어 (NGCFLayer) 정규화 및 메시지 전달

### 핵심 원리 및 코드 설명
- **NGCF(Neural Graph Collaborative Filtering) 메시지 전달 수식**:
  - 이웃 아이템 $i$에서 사용자 $u$로 전달되는 메시지는 단순 선형 변환 $W_1 e_i$뿐만 아니라 사용자-아이템 간 요소별 곱(아다마르 곱 $e_i \odot e_u$)을 반영하여 친화도가 높은 이웃의 신호를 증폭합니다:
    $$m_{u \leftarrow i} = \frac{1}{\sqrt{|N_u||N_i|}} \left( W_1 e_i + W_2 (e_i \odot e_u) \right)$$
  - 차수 정규화 항 $\frac{1}{\sqrt{\text{deg}(u)\text{deg}(i)}}$는 연결된 엣지가 많은 인기 노드의 메시지 크기가 폭발하는 것을 방지합니다.

### 🎯 빈칸(### 작성 필요 ###)별 출제 포인트 & 파라미터 역할
1. **빈칸 1 (`1.0/torch.sqrt(deg[src]*deg[dst])`)**:
   - **출제 포인트**: 각 엣지의 출발 노드 차수와 도착 노드 차수의 기하평균 역수로 정규화 가중치를 계산합니다.
2. **빈칸 2 (`self.W1(dst_feat) + self.W2(dst_feat*src_feat)`)**:
   - **출제 포인트**: 이웃 특징의 선형 변환(`W1`)과 두 노드 특징 간 요소별 곱의 선형 변환(`W2`)을 더하여 상호작용 메시지를 구성합니다.
3. **빈칸 3 (`aggregated_messages.index_add_(0, src, edge_messages_for_src)`)**:
   - **출제 포인트**: 0번(노드) 차원을 기준으로 각 엣지의 출발 노드 인덱스(`src`) 위치에 메시지를 합산(scatter-add)합니다.

In [ ]:
# [코드]
class NGCFLayer(nn.Module):
    def __init__(self, input_dim, output_dim, dropout=0.1):
        super().__init__()
        self.W1 = nn.Linear(input_dim, output_dim)
        self.W2 = nn.Linear(input_dim, output_dim)
        self.leaky_relu = nn.LeakyReLU(0.2)

    def forward(self, edge_index, node_features, user_num, item_num):
        src, dst = edge_index # src : user, dst : movie

        # 각 노드의 연결 차수(degree) 계산
        deg = torch.zeros(node_features.size(0), device=node_features.device)
        deg.index_add_(0, src, torch.ones_like(src, dtype=torch.float))
        deg.index_add_(0, dst, torch.ones_like(dst, dtype=torch.float))

        # 라플라시안 정규화 계수 1 / sqrt(deg(u) * deg(i)) 계산
        norm = ### 1.0/torch.sqrt(deg[src]*deg[dst]) ### 작성 필요 ###

        src_feat = node_features[src] # H_u
        dst_feat = node_features[dst] # H_i

        # 이웃 특징(W1)과 상호작용 요소별 곱 특징(W2)을 결합하여 메시지 생성
        edge_messages_for_src = ### self.W1(dst_feat) + self.W2(dst_feat*src_feat) ### 작성 필요 ###
        edge_messages_for_src *= norm.unsqueeze(1)

        edge_messages_for_dst = self.W1(src_feat) + self.W2(src_feat*dst_feat)
        edge_messages_for_dst *= norm.unsqueeze(1)

        # 노드별로 수신된 엣지 메시지 집계(index_add_) 및 자기 자신 변환(W1) 합산
        aggregated_messages = torch.zeros_like(node_features)
        ### aggregated_messages.index_add_(0, src, edge_messages_for_src) ### 작성 필요 ###
        aggregated_messages[:user_num] += self.W1(node_features[:user_num])

        aggregated_messages.index_add_(0, dst, edge_messages_for_dst)
        aggregated_messages[user_num:] += self.W1(node_features[user_num:])

        aggregated_features = self.leaky_relu(aggregated_messages)
        return aggregated_features

## 핵심 개념 17. (c-3) NGCF 다층 표현 결합(Concatenation) 및 BPR Loss 구현

### 핵심 원리 및 코드 설명
- **다층 표현 결합 (Layer-wise Concatenation)**: NGCF는 0층(초기 임베딩), 1층(1-hop 이웃), 2층(2-hop 이웃)의 표현을 모두 보존하기 위해 각 층의 출력을 특징 축(`dim=-1`)으로 이어 붙입니다: $e_u^* = e_u^{(0)} \Vert e_u^{(1)} \Vert \dots \Vert e_u^{(L)}$.
- **BPR (Bayesian Personalized Ranking) 손실**: 관측된 긍정 아이템($i$)의 내적 점수 $\hat{y}_{ui}$가 관측되지 않은 부정 아이템($j$)의 내적 점수 $\hat{y}_{uj}$보다 크도록 쌍별(Pairwise) 순위를 최적화합니다:
  $$L_{\text{BPR}} = -\frac{1}{N}\sum \ln \sigma(\hat{y}_{ui} - \hat{y}_{uj}) + \lambda \|\Theta\|_2^2$$

### 🎯 빈칸(### 작성 필요 ###)별 출제 포인트 & 파라미터 역할
1. **빈칸 1 (`torch.concat(layer_outputs,dim=-1)`)**:
   - **출제 포인트**: 각 레이어의 노드 특징 행렬들을 마지막 임베딩 차원(`dim=-1`)으로 이어 붙입니다.
2. **빈칸 2 (`final_features[:self.num_users]`)**:
   - **출제 포인트**: 전체 노드 행렬에서 앞쪽 `num_users`개의 사용자 노드 표현만 슬라이싱합니다.
3. **빈칸 3 (`-torch.mean(F.logsigmoid(pos_scores - neg_scores))`)**:
   - **출제 포인트**: 긍정 점수와 부정 점수 차이에 로그 시그모이드를 적용하고 음의 평균을 취해 BPR 손실을 계산합니다.

In [ ]:
# [코드]
class NGCF(nn.Module):
    def __init__(self, num_users, num_items, embedding_dim, layer_dims, dropout=0.1):
        super().__init__()
        self.num_users = num_users
        self.num_items = num_items
        self.embedding_dim = embedding_dim
        self.node_embeddings = nn.Embedding(self.num_users+self.num_items,self.embedding_dim)
        nn.init.xavier_uniform_(self.node_embeddings.weight)

        self.layers = nn.ModuleList([
            NGCFLayer(input_dim=(embedding_dim if i == 0 else layer_dims[i - 1]),
                      output_dim=layer_dims[i], dropout=dropout)
            for i in range(len(layer_dims))
        ])

    def forward(self, edge_index):
        node_features = self.node_embeddings.weight
        layer_outputs = [node_features]
        for layer in self.layers:
            node_features = layer(edge_index, node_features,self.num_users, self.num_items)
            layer_outputs.append(node_features)

        # 초기 임베딩과 각 레이어의 출력 임베딩을 마지막 차원(dim=-1)으로 결합
        final_features = ### torch.concat(layer_outputs,dim=-1) ### 작성 필요 ###
        # 앞쪽 num_users개 행을 사용자 특징으로, 나머지를 아이템 특징으로 분리
        user_features = ### final_features[:self.num_users] ### 작성 필요 ###
        item_features = final_features[self.num_users:]
        return user_features, item_features

    def bpr_loss(self, user_emb, pos_item_emb, neg_item_emb, reg_weight=1e-4):
        # 사용자-긍정 아이템 내적 점수 및 사용자-부정 아이템 내적 점수 계산
        pos_scores = torch.sum(user_emb * pos_item_emb, dim=1)
        neg_scores = torch.sum(user_emb * neg_item_emb, dim=1)
        # BPR 쌍별 순위 손실: -mean(log(sigmoid(pos_scores - neg_scores)))
        loss = ### -torch.mean(F.logsigmoid(pos_scores - neg_scores)) ### 작성 필요 ###
        # L2 정규화 손실 추가
        reg_loss = reg_weight * (user_emb.norm(2).pow(2) + pos_item_emb.norm(2).pow(2)                                  + neg_item_emb.norm(2).pow(2)) / user_emb.size(0)
        return loss + reg_loss

## 핵심 개념 18. (c-4) NGCF 추천 랭킹 평가 함수 구현 (Top-K 추출 및 NDCG@K)

### 핵심 원리 및 코드 설명
- **NDCG@K (Normalized Discounted Cumulative Gain)**: 상위 순위에 정답 아이템이 배치될수록 더 높은 가중치를 부여하는 랭킹 지표입니다.
  - 0-기반 인덱스 `rank`($0, 1, \dots, k-1$)에 대해 로그 할인 분모는 $\log_2(\text{rank} + 2)$가 됩니다:
    $$\text{DCG@K} = \sum_{r=0}^{K-1} \frac{\mathbb{I}(\text{item}_r \in \text{Pos})}{\log_2(r + 2)}, \quad \text{IDCG@K} = \sum_{r=0}^{\min(|\text{Pos}|, K)-1} \frac{1}{\log_2(r + 2)}, \quad \text{NDCG@K} = \frac{\text{DCG@K}}{\text{IDCG@K}}$$

### 🎯 빈칸(### 작성 필요 ###)별 출제 포인트 & 파라미터 역할
1. **빈칸 1 (`torch.topk(scores, k=k)`)**:
   - **출제 포인트**: 전체 아이템 점수 벡터에서 가장 높은 상위 `k`개의 값과 인덱스를 동시에 추출합니다.
2. **빈칸 2 (`1.0 / math.log2(rank + 2)`)**:
   - **출제 포인트**: 0부터 시작하는 순위 `rank`에 대해 $\log_2(\text{rank}+2)$로 나누어 순위가 낮아질수록 이득을 로그 스케일로 할인합니다.
3. **빈칸 3 (`dcg / idcg if idcg > 0 else 0.0`)**:
   - **출제 포인트**: 실제 DCG를 이상적인 최대 DCG(IDCG)로 나누어 `[0, 1]` 범위의 NDCG로 정규화합니다.

In [ ]:
# [코드]
def evaluate(user_features, item_features, test_edge_index, k):
    user_pos_items = defaultdict(list)
    E_test = test_edge_index.size(1)
    for i in range(E_test):
        u = test_edge_index[0, i].item()
        it = test_edge_index[1, i].item() - num_users
        user_pos_items[u].append(it)

    recalls, precisions, ndcgs = [], [], []
    for user, pos_items in user_pos_items.items():
        user_emb = user_features[user]
        # 전체 아이템 특징과 사용자 임베딩의 내적으로 추천 점수 계산
        scores = torch.matmul(item_features, user_emb)
        # 점수가 가장 높은 상위 k개 아이템의 점수와 인덱스 추출
        topk_scores, topk_indices = ### torch.topk(scores, k=k) ### 작성 필요 ###
        topk_indices = topk_indices.cpu().numpy().tolist()

        hits = 0
        dcg = 0.0
        idcg = 0.0
        n_pos = len(pos_items)

        for rank, item_idx in enumerate(topk_indices):
            if item_idx in pos_items:
                hits += 1
                # 적중한 아이템의 순위(rank)에 따른 로그 할인 이득(DCG) 누적
                dcg += ### 1.0 / math.log2(rank + 2) ### 작성 필요 ###
        for rank in range(min(n_pos, k)):
            idcg += 1.0 / math.log2(rank + 2)

        recall_u = hits / n_pos
        precision_u = hits / k
        # 이상적인 최대 이득(IDCG) 대비 실제 DCG 비율로 NDCG 계산
        ndcg_u = ### dcg / idcg if idcg > 0 else 0.0 ### 작성 필요 ###

        recalls.append(recall_u)
        precisions.append(precision_u)
        ndcgs.append(ndcg_u)

    recall = np.mean(recalls)
    precision = np.mean(precisions)
    ndcg = np.mean(ndcgs)
    return recall, precision, ndcg